In [3]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

model = init_chat_model(model="gpt-5-nano")
load_dotenv()

True

In [4]:
agent = create_agent(
    model= model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger=("tokens",100),
            keep=("messages",1)
        )
    ] ,
)

In [5]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n- Capital of the Moon: Lunapolis\n- Weather: clear skies; high 120°C, low -100°C\n- Population: 100,000 cheese miners\n- Labor: Cheese miners' union likely to strike due to dissatisfaction with the new president", additional_kwargs={}, response_metadata={}, id='c2ba6518-f3d2-4b7b-97f8-029c166809e0'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='27de7289-1cb7-4168-a118-6f76d2971fb1'),
              AIMessage(content='In a fictional Lunapolis, here’s a constructive, non-confrontational way I’d respond as the new president to the cheese miners’ union. The focus is safety, fair treatment, and getting to a real, lasting agreement through open, data-driven negotiation.\n\nKey principles\n- Acknowledge the miners’ contributions and right to fair treatment.\n- Prioritize safety in e

In [7]:
print(response["messages"][-1].content)

In a fictional Lunapolis, here’s a constructive, non-confrontational way I’d respond as the new president to the cheese miners’ union. The focus is safety, fair treatment, and getting to a real, lasting agreement through open, data-driven negotiation.

Key principles
- Acknowledge the miners’ contributions and right to fair treatment.
- Prioritize safety in extreme lunar conditions (heat, cold, oxygen, exposure).
- Engage in good-faith, transparent, interest-based bargaining.
- Keep essential cheese production going for public needs while negotiations continue.
- Use independent mediation if needed to break deadlocks.

Immediate actions (first 72 hours)
- Publicly acknowledge the concerns and commit to safe, fair working conditions and a rapid negotiation path.
- Invite union leadership to a joint, scheduled negotiation session with a neutral facilitator (and, if desired, a mediator from an external labor-relations body).
- Establish a temporary, non-retaliatory moratorium on disciplin

In [8]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage, ToolMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent


In [9]:
@before_agent
def trim_messages(state: AgentState, runtime: Runtime)->dict[str,Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m,ToolMessage)]

    return {"messages":[RemoveMessage(id=m.id) for m in tool_messages]}

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [10]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='4d4a91e7-ab33-4ff1-87ce-81e72d869b99'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='144bd3c7-41b8-4ed1-ab41-89bdcb467460', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='f342f773-30cb-48a3-a4bb-1abc9b0caa0b'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='ee9deba9-2f07-4d96-888f-36d9b170cda9', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='b50479ea-9885-40c2-a159-b2037e7c534f'),
              AIMessage(content='I can’t read the device’s temperature from here. If you want t

In [11]:
print(response["messages"][-1].content)

I can’t read the device’s temperature from here. If you want to check it yourself, tell me what type of device you’re using (PC/laptop, smartphone, tablet, etc.) and the model, and I’ll give precise steps. For now, here are general options and safety tips:

What to do if it’s overheating or won’t turn on
- If it feels very hot to touch, power off and unplug it. Let it cool in a ventilated area before you try to power it on again.
- Check for blocked vents or a dusty fan. If you can safely do so, clean the vents with compressed air.
- Remove any case or cover that might be trapping heat (only if you can do it safely).

How to check temperature on common devices
- Windows PC/Laptop:
  - Built-in: Task Manager > Performance > CPU (and GPU) to see activity, but not exact temps. 
  - Third-party: HWMonitor, Core Temp, or HWInfo show CPU/GPU temps.
- Mac:
  - macOS doesn’t show temps by default. Use third-party apps like iStat Menus or Macs Fan Control to read temps and fan speeds.
- Linux:


In [18]:
from langchain.agents import create_agent, AgentState
from langchain.tools import ToolRuntime,tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model

@tool
def read_email(runtime: ToolRuntime) ->str:
    "Reads an email from given address"
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body"""
    return f"Email sent"

model = init_chat_model("gpt-5-nano")
class EmailState(AgentState):
    email: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [19]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id":"1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [20]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads '
                                                                          'up. '
                                                                          'I’m '
                                                                          'happy '
                                                                          'to '
               

In [21]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for the heads up. I’m happy to reschedule. Are any of these times tomorrow convenient?

- 10:00 AM
- 1:30 PM
- 4:00 PM

If none of these work, please suggest a couple of alternative times and I’ll adapt.

Best regards,
Seán


In [22]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='791ff02e-a668-4d3f-908d-0fae14130b11'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 467, 'prompt_tokens': 166, 'total_tokens': 633, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DSCihqgXF077rjuZnkyVJdXyQ0VWm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6ade-5d1a-7e51-8b4f-a41ec826bb39-0', tool_calls

In [23]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='791ff02e-a668-4d3f-908d-0fae14130b11'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 467, 'prompt_tokens': 166, 'total_tokens': 633, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DSCihqgXF077rjuZnkyVJdXyQ0VWm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6ade-5d1a-7e51-8b4f-a41ec826bb39-0', tool_calls

In [24]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='791ff02e-a668-4d3f-908d-0fae14130b11'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 467, 'prompt_tokens': 166, 'total_tokens': 633, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DSCihqgXF077rjuZnkyVJdXyQ0VWm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6ade-5d1a-7e51-8b4f-a41ec826bb39-0', tool_calls